In [8]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
# ── Pluviometer data ──────────────────────────────────────────────────────────
# Excel files have two header rows; row 0 = internal field names, row 1 = labels.
# We skip both and name columns manually.

pluvio_files = sorted(glob.glob("Pluviometer data/*.xlsx"))

pluvio_frames = []
for f in pluvio_files:
    df = pd.read_excel(
        f,
        header=1,          # row 1 (0-indexed) has the real column labels
        usecols=[0, 1, 2], # we need just the first three columns (datetime, Avant-Port, Flagey)
    )
    df.columns = ["datetime", "Avant-Port_mm", "Flagey_mm"]
    pluvio_frames.append(df)
    print(f"File: {f}  →  Shape: {df.shape}")

File: Pluviometer data\2022.xlsx  →  Shape: (105120, 3)
File: Pluviometer data\2023.xlsx  →  Shape: (105120, 3)
File: Pluviometer data\2024.xlsx  →  Shape: (105421, 3)


In [3]:
pluvio_frames

[                  datetime  Avant-Port_mm  Flagey_mm
 0      2022-01-01 00:00:00            0.0        0.0
 1      2022-01-01 00:05:00            0.0        0.0
 2      2022-01-01 00:10:00            0.0        0.0
 3      2022-01-01 00:15:00            0.0        0.0
 4      2022-01-01 00:20:00            0.0        0.0
 ...                    ...            ...        ...
 105115 2022-12-31 23:35:00            0.0        0.0
 105116 2022-12-31 23:40:00            0.0        0.0
 105117 2022-12-31 23:45:00            0.0        0.0
 105118 2022-12-31 23:50:00            0.0        0.0
 105119 2022-12-31 23:55:00            0.0        0.0
 
 [105120 rows x 3 columns],
                   datetime  Avant-Port_mm  Flagey_mm
 0      2023-01-01 00:00:00            0.0        0.0
 1      2023-01-01 00:05:00            0.0        0.0
 2      2023-01-01 00:10:00            0.0        0.0
 3      2023-01-01 00:15:00            0.0        0.0
 4      2023-01-01 00:20:00            0.0        0.

In [4]:
# Combine all pluvio frames into one DataFrame and reset the index
rain = pd.concat(pluvio_frames, ignore_index=True)
print(f"Combined shape: {rain.shape}")

Combined shape: (315661, 3)


In [5]:
# Parse datetime; coerce summary rows (TOTAL, JAN, …) to NaT
rain["datetime"] = pd.to_datetime(rain["datetime"], errors="coerce")
# Drop rows with invalid datetime (NaN or NaT)
rain = rain.dropna(subset=["datetime"])
# Set datetime as index and sort
rain = rain.set_index("datetime").sort_index()

# Convert to numeric; coerce non-numeric values (e.g. "TOTAL") to NaN
rain["Avant-Port_mm"] = pd.to_numeric(rain["Avant-Port_mm"], errors="coerce")
rain["Flagey_mm"] = pd.to_numeric(rain["Flagey_mm"], errors="coerce")

print("=" * 60)
print("PLUVIOMETER DATA")
print("=" * 60)
print(f"Shape     : {rain.shape}")
print(f"Date range: {rain.index.min()}  to  {rain.index.max()}")
print(f"Freq hint : {pd.infer_freq(rain.index[:200])}")
print("\nHead:")
print(rain.head())
print("\ndtypes:")
print(rain.dtypes)
print(f"\nMissing values:\n{rain.isna().sum()}")

PLUVIOMETER DATA
Shape     : (315648, 2)
Date range: 2022-01-01 00:00:00  to  2024-12-31 23:55:00
Freq hint : 5min

Head:
                     Avant-Port_mm  Flagey_mm
datetime                                     
2022-01-01 00:00:00            0.0        0.0
2022-01-01 00:05:00            0.0        0.0
2022-01-01 00:10:00            0.0        0.0
2022-01-01 00:15:00            0.0        0.0
2022-01-01 00:20:00            0.0        0.0

dtypes:
Avant-Port_mm    float64
Flagey_mm        float64
dtype: object

Missing values:
Avant-Port_mm    0
Flagey_mm        0
dtype: int64


In [6]:
# ── Sewage data ───────────────────────────────────────────────────────────────
sewage_files = sorted(glob.glob("Sewage data/*.csv"))

sewage_frames = []
for f in sewage_files:
    df = pd.read_csv(f, parse_dates=["Date"])
    sewage_frames.append(df)
    print(f"File: {f}  →  Shape: {df.shape}")

sewer = pd.concat(sewage_frames, ignore_index=True)
sewer = sewer.rename(columns={"Date": "datetime", "Value": "level_mm"})
sewer["datetime"] = pd.to_datetime(sewer["datetime"])
sewer = sewer.set_index("datetime").sort_index()

print("\n" + "=" * 60)
print("SEWAGE DATA  (DO08 — Lion overflow)")
print("=" * 60)
print(f"Shape     : {sewer.shape}")
print(f"Date range: {sewer.index.min()}  to  {sewer.index.max()}")
print("\nHead:")
print(sewer.head())
print("\ndtypes:")
print(sewer.dtypes)
print(f"\nMissing values:\n{sewer.isna().sum()}")
print(f"\nOverflow events (level > 3000 mm): {(sewer['level_mm'] > 3000).sum()}")

File: Sewage data\U24_2022.csv  →  Shape: (524147, 2)
File: Sewage data\U24_2023.csv  →  Shape: (524154, 2)
File: Sewage data\U24_2024.csv  →  Shape: (525599, 2)

SEWAGE DATA  (DO08 — Lion overflow)
Shape     : (1573900, 1)
Date range: 2022-01-01 00:00:33  to  2024-12-30 23:59:33

Head:
                     level_mm
datetime                     
2022-01-01 00:00:33   883.908
2022-01-01 00:01:34   881.589
2022-01-01 00:02:33   874.169
2022-01-01 00:03:33   876.719
2022-01-01 00:04:33   876.719

dtypes:
level_mm    float64
dtype: object

Missing values:
level_mm    0
dtype: int64

Overflow events (level > 3000 mm): 30034


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pre-aggregate full-series data to hourly BEFORE plotting
# Reduces 1.57M sewer points -> ~26K and 315K rain points -> ~26K
sewer_h = sewer["level_mm"].resample("1h").mean()
rain_h  = rain.resample("1h").sum()

zoom_start, zoom_end = "2022-06-01", "2022-06-10"
sewer_zoom = sewer.loc[zoom_start:zoom_end]
rain_zoom  = rain.loc[zoom_start:zoom_end]

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=(
        "Full Sewer Level (hourly avg)", "",
        "Sewer Level Distribution", "Rainfall Distribution (non-zero)",
        f"Sewer Zoom: {zoom_start} to {zoom_end}", f"Rainfall Zoom: {zoom_start} to {zoom_end}",
        "Sewer Level per Year (box)", "",
    ),
    specs=[
        [{"colspan": 2}, None],
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None],
    ],
    vertical_spacing=0.09,
    horizontal_spacing=0.08,
)

# 1. Full sewer -- hourly avg, WebGL-accelerated
fig.add_trace(go.Scattergl(
    x=sewer_h.index, y=sewer_h.values,
    mode="lines", line=dict(color="steelblue", width=0.8),
    name="DO08 level (hourly avg)",
), row=1, col=1)
fig.add_hline(y=3000, line_dash="dash", line_color="red",
              annotation_text="Overflow (3000 mm)", row=1, col=1)

# 2. Sewer histogram (sampled to 100K for speed)
sample = sewer["level_mm"].dropna().sample(n=min(100_000, len(sewer)), random_state=0)
fig.add_trace(go.Histogram(x=sample, nbinsx=80, marker_color="steelblue",
                           name="Sewer dist", showlegend=False), row=2, col=1)
fig.add_vline(x=3000, line_dash="dash", line_color="red", row=2, col=1)

# 3. Rainfall histogram (non-zero only)
fig.add_trace(go.Histogram(
    x=rain["Avant-Port_mm"][rain["Avant-Port_mm"] > 0.01].dropna(),
    nbinsx=50, marker_color="royalblue", opacity=0.6, name="Avant-Port", showlegend=False,
), row=2, col=2)
fig.add_trace(go.Histogram(
    x=rain["Flagey_mm"][rain["Flagey_mm"] > 0.01].dropna(),
    nbinsx=50, marker_color="darkorange", opacity=0.6, name="Flagey", showlegend=False,
), row=2, col=2)

# 4. Zoom -- sewer (full 5-min resolution, small window ~2880 pts)
fig.add_trace(go.Scatter(
    x=sewer_zoom.index, y=sewer_zoom["level_mm"],
    mode="lines", line=dict(color="steelblue", width=1.2),
    name="Sewer zoom", showlegend=False,
), row=3, col=1)
fig.add_hline(y=3000, line_dash="dash", line_color="red", row=3, col=1)

# 5. Zoom -- rainfall (area fill instead of individual bars -- much faster)
fig.add_trace(go.Scatter(
    x=rain_zoom.index, y=rain_zoom["Avant-Port_mm"],
    mode="lines", fill="tozeroy", line=dict(color="royalblue", width=0.5),
    name="Avant-Port zoom", showlegend=False,
), row=3, col=2)
fig.add_trace(go.Scatter(
    x=rain_zoom.index, y=rain_zoom["Flagey_mm"],
    mode="lines", fill="tozeroy", line=dict(color="darkorange", width=0.5),
    name="Flagey zoom", showlegend=False,
), row=3, col=2)

# 6. Box plots per year (hourly data -- statistically representative, much smaller)
for year, color in [("2022", "steelblue"), ("2023", "mediumseagreen"), ("2024", "mediumpurple")]:
    fig.add_trace(go.Box(
        y=sewer_h.loc[year].dropna(), name=year,
        marker_color=color, boxmean=True, showlegend=False,
    ), row=4, col=1)
fig.add_hline(y=3000, line_dash="dash", line_color="red", row=4, col=1)

fig.update_layout(
    height=1400, barmode="overlay", template="plotly_white",
    title=dict(text="Raw Data Exploration", x=0.5, font=dict(size=16)),
)
fig.update_yaxes(title_text="Level (mm)",     row=1, col=1)
fig.update_yaxes(title_text="Frequency",      row=2, col=1)
fig.update_yaxes(title_text="Frequency",      row=2, col=2)
fig.update_yaxes(title_text="Level (mm)",     row=3, col=1)
fig.update_yaxes(title_text="Rain (mm/5min)", row=3, col=2)
fig.update_yaxes(title_text="Level (mm)",     row=4, col=1)

fig.show()
fig.write_html("raw_data_exploration.html")
print("Saved: raw_data_exploration.html")


# Step 2 — Raw Data Cleaning & Merging

In [ ]:
import numpy as np

# ── Constants ─────────────────────────────────────────────────────────────────────────
PHYSICAL_MIN     =  0.0  # mm -- below 0 is impossible
MAX_INTERP_STEPS =  6    # 6 x 5 min = 30 min max gap to interpolate

# ── 2a: Remove physically impossible values (negatives) ──────────────────
sewer_clean = sewer.copy()
n_neg = (sewer_clean["level_mm"] < PHYSICAL_MIN).sum()
sewer_clean.loc[sewer_clean["level_mm"] < PHYSICAL_MIN, "level_mm"] = np.nan
print(f"Removed {n_neg:,} values below {PHYSICAL_MIN} mm (negatives)")


In [ ]:
# ── 2c: Resample sewer from 1-min to 5-min (mean of clean readings) ───────────
sewer_5min = (
    sewer_clean["level_mm"]
    .resample("5min")
    .mean()
    .rename("level_mm")
    .to_frame()
)
print(f"Shape after resample : {sewer_5min.shape}")
print(f"NaN bins after resample: {sewer_5min['level_mm'].isna().sum():,}")

# ── 2d: Interpolate gaps up to 30 min ────────────────────────────────────────
nan_before = sewer_5min["level_mm"].isna()
sewer_5min["level_mm"] = sewer_5min["level_mm"].interpolate(
    method="time", limit=MAX_INTERP_STEPS, limit_direction="forward"
)
nan_after = sewer_5min["level_mm"].isna()
print(f"Interpolated : {(nan_before & ~nan_after).sum():,} steps (gaps <= {MAX_INTERP_STEPS * 5} min)")
print(f"Remaining NaN (long gaps): {nan_after.sum():,} steps")

In [ ]:
# ── 2e: Clean rainfall (clip negatives defensively) ──────────────────────────
rain_clean = rain.copy()
for col in ["Avant-Port_mm", "Flagey_mm"]:
    n_neg = (rain_clean[col] < 0).sum()
    rain_clean.loc[rain_clean[col] < 0, col] = np.nan
    print(f"Removed {n_neg} negative values in {col}")

# ── 2f: Merge on common 5-min index ──────────────────────────────────────────
df = sewer_5min.join(rain_clean, how="inner")

print(f"\nMerged shape : {df.shape}")
print(f"Date range   : {df.index.min()}  to  {df.index.max()}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDescribe:\n{df.describe().round(3)}")

In [ ]:
def count_overflow_events(series, threshold=3000):
    """Count distinct overflow episodes (consecutive readings above threshold)"""
    is_overflow = series > threshold
    events = (is_overflow & ~is_overflow.shift(1).fillna(False)).sum()
    return events

n_events_original  = count_overflow_events(sewer_clean["level_mm"])
n_events_resampled = count_overflow_events(merged["level_mm"])

print("=" * 50)
print("CORRECT OVERFLOW PRESERVATION CHECK")
print("=" * 50)
print(f"\nOverflow EVENTS (distinct episodes):")
print(f"  Original  (1-min): {n_events_original:,}  events")
print(f"  Resampled (5-min): {n_events_resampled:,}  events")
print(f"  Event retention  : {n_events_resampled/n_events_original*100:.1f}%")
print(f"\nOverflow DURATION (total time above 3000mm):")
duration_original  = (sewer_clean["level_mm"] > 3000).sum() * 1
duration_resampled = (merged["level_mm"] > 3000).sum() * 5
print(f"  Original  (1-min): {duration_original:,}  minutes  = {duration_original/60:.1f} hours")
print(f"  Resampled (5-min): {duration_resampled:,}  minutes  = {duration_resampled/60:.1f} hours")
print(f"  Duration retention: {duration_resampled/duration_original*100:.1f}%")
print(f"\nConclusion:")
print(f"  30,034 readings at 1-min / 5 = {30034//5:,} expected 5-min readings")
print(f"  Actual 5-min overflow readings: {n_overflow_resampled:,}")
print(f"  Resolution change is mathematically consistent")


In [ ]:
# ── 2e: Build overflow events table from raw 1-min cleaned data ─────────────────
is_overflow  = sewer_clean["level_mm"] > OVERFLOW_THRESHOLD
event_id     = (is_overflow & ~is_overflow.shift(1).fillna(False)).cumsum()
event_id     = event_id.where(is_overflow)

events_raw = (
    sewer_clean.assign(event_id=event_id)
    .dropna(subset=["event_id"])
    .groupby("event_id")
    .agg(
        start    =("level_mm", lambda x: x.index.min()),
        end      =("level_mm", lambda x: x.index.max()),
        peak_mm  =("level_mm", "max"),
        mean_mm  =("level_mm", "mean"),
    )
    .reset_index(drop=True)
)
events_raw["duration_min"] = (
    (events_raw["end"] - events_raw["start"])
    .dt.total_seconds()
    .div(60)
    .add(1)
    .astype(int)
)
print(f"Raw overflow events detected: {len(events_raw)}")

# ── 2f: Gap sensitivity analysis ───────────────────────────────────────────────────
def count_events_for_gap(events_df, gap_min):
    gap = pd.Timedelta(minutes=gap_min)
    df_ = events_df.sort_values("start").reset_index(drop=True)
    out, current = [], df_.iloc[0].to_dict()
    for _, row in df_.iloc[1:].iterrows():
        if row["start"] - current["end"] <= gap:
            current["end"]     = row["end"]
            current["peak_mm"] = max(current["peak_mm"], row["peak_mm"])
        else:
            out.append(current)
            current = row.to_dict()
    out.append(current)
    return len(out)

print("\nGap sensitivity analysis:")
print("  Gap threshold -> event count")
for g in [0, 5, 10, 15, 20, 30, 60]:
    print(f"    {g:>3} min  ->  {count_events_for_gap(events_raw, g)} events")

# ── 2g: Merge events with gap < 10 min ──────────────────────────────────────────
MIN_DRY_GAP = pd.Timedelta(minutes=10)
merged_list = []
current     = events_raw.iloc[0].to_dict()

for _, row in events_raw.iloc[1:].iterrows():
    if row["start"] - current["end"] <= MIN_DRY_GAP:
        current["end"]          = row["end"]
        current["peak_mm"]      = max(current["peak_mm"], row["peak_mm"])
        current["duration_min"] = int(
            (current["end"] - current["start"]).total_seconds() / 60
        ) + 1
    else:
        merged_list.append(current)
        current = row.to_dict()

merged_list.append(current)
events_merged = pd.DataFrame(merged_list).reset_index(drop=True)

def get_mean(start, end):
    mask = (
        (sewer_clean.index >= start) &
        (sewer_clean.index <= end)   &
        (sewer_clean["level_mm"] > OVERFLOW_THRESHOLD)
    )
    return sewer_clean.loc[mask, "level_mm"].mean()

events_merged["mean_mm"]  = events_merged.apply(lambda r: get_mean(r["start"], r["end"]), axis=1)
events_merged["duration"] = events_merged["duration_min"].apply(
    lambda m: f"{m//60}h {m%60}min" if m >= 60 else f"{m}min"
)

print(f"\n{'='*55}")
print(f"OVERFLOW EVENT MERGING SUMMARY")
print(f"{'='*55}")
print(f"  Gap tolerance         : {MIN_DRY_GAP}")
print(f"  Events before merging : {len(events_raw)}")
print(f"  Events after merging  : {len(events_merged)}")
print(f"  Events merged away    : {len(events_raw) - len(events_merged)}")
print()
print(events_merged[["start", "end", "duration", "peak_mm", "mean_mm"]].to_string())

events_merged.to_csv("overflow_events_merged.csv", index=False)
print("\nSaved: overflow_events_merged.csv")


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Downsample to hourly for the overview so it renders quickly
df_plot = df.resample("1h").mean()

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        "DO08 sewer level (hourly avg for overview)",
        "Rainfall — Avant-Port",
        "Rainfall — Flagey",
    ],
    vertical_spacing=0.07,
)

fig.add_trace(
    go.Scatter(x=df_plot.index, y=df_plot["level_mm"],
               mode="lines", line=dict(color="steelblue", width=1), name="DO08 level"),
    row=1, col=1,
)
fig.add_hline(y=3000, line_dash="dash", line_color="red",
              annotation_text="Overflow threshold (3000 mm)",
              annotation_position="top right", row=1, col=1)
fig.add_trace(
    go.Bar(x=df_plot.index, y=df_plot["Avant-Port_mm"],
           marker_color="cornflowerblue", name="Avant-Port"),
    row=2, col=1,
)
fig.add_trace(
    go.Bar(x=df_plot.index, y=df_plot["Flagey_mm"],
           marker_color="mediumseagreen", name="Flagey"),
    row=3, col=1,
)
fig.update_yaxes(title_text="Level (mm)", row=1, col=1)
fig.update_yaxes(title_text="Rain (mm/h)", row=2, col=1)
fig.update_yaxes(title_text="Rain (mm/h)", row=3, col=1)
fig.update_layout(height=750, title_text="Full time-series overview — cleaned data")
fig.show()

In [ ]:
# ── Zoom: 12-hour window around a representative overflow event ───────────────
overflow_times = df.index[df["level_mm"] > 3000]
event_center = overflow_times[len(overflow_times) // 2]
zoom = df.loc[event_center - pd.Timedelta(hours=6) : event_center + pd.Timedelta(hours=6)]

fig2 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[
        f"Sewer level — event centred on {event_center.strftime('%Y-%m-%d %H:%M')}",
        "Rainfall (5-min)",
    ],
    vertical_spacing=0.1,
)
fig2.add_trace(
    go.Scatter(x=zoom.index, y=zoom["level_mm"],
               mode="lines", line=dict(color="steelblue", width=2), name="DO08 level"),
    row=1, col=1,
)
fig2.add_hline(y=3000, line_dash="dash", line_color="red",
               annotation_text="3000 mm", row=1, col=1)
fig2.add_trace(
    go.Bar(x=zoom.index, y=zoom["Avant-Port_mm"], name="Avant-Port",
           marker_color="cornflowerblue"),
    row=2, col=1,
)
fig2.add_trace(
    go.Bar(x=zoom.index, y=zoom["Flagey_mm"], name="Flagey",
           marker_color="mediumseagreen"),
    row=2, col=1,
)
fig2.update_yaxes(title_text="Level (mm)", row=1, col=1)
fig2.update_yaxes(title_text="Rain (mm/5min)", row=2, col=1)
fig2.update_layout(height=600, barmode="overlay")
fig2.show()

In [ ]:
# ── Save cleaned merged dataset for downstream steps ─────────────────────────
df.to_parquet("data_clean.parquet")
print(f"Saved: data_clean.parquet  |  shape: {df.shape}  |  columns: {df.columns.tolist()}")